In [1]:
import pathlib

import yaml
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr

import torch

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

import ipywidgets as widgets
from sklearn.manifold import TSNE

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = 'small'
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
len(ds)

12983

*Base* file name for the results (trained model)

In [6]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_50_300_epochs' 


In [7]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [8]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [9]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [10]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
parameters

{'PRNG seeds': {'model initialization': {'decoder': 43, 'encoder': 42},
  'sampling': {'training set': 14, 'validation set': 15},
  'training dataloader': 43,
  'training-validation splitting': 42},
 'encoder': {'input_size': [1, 10000]},
 'loss': {'x_sd': 0.25},
 'model': {'channels': [12, 12, 12],
  'dim_z': 50,
  'kernel_sizes': [128, 64, 32],
  'stride': 2},
 'training': {'batch_size': 32,
  'dl_rng_key': 43,
  'learning_rate': 3e-05,
  'n_epochs': 300,
  'train_test_split': {'test_size': 0.2}}}

In [11]:
results_df = pd.read_parquet(results_file)
results_df

training                              validation                \
             loss loglikelihood          kl          loss loglikelihood   
0    53285.791412 -53123.421280  162.370163  30294.680807 -30027.265006   
1    27905.011719 -27593.666367  311.345325  26352.327696 -26004.388481   
2    25648.908918 -25288.733212  360.175734  25146.346608 -24773.675877   
3    24702.890757 -24329.036472  373.854259  24469.998499 -24091.930688   
4    24190.128867 -23815.185685  374.943163  24065.801996 -23692.647771   
..            ...           ...         ...           ...           ...   
295  18379.622800 -18133.303642  246.319160  19344.585604 -19101.085640   
296  18392.093422 -18146.806379  245.287084  19240.253216 -18997.212140   
297  18384.647581 -18139.175162  245.472415  19334.408787 -19090.609685   
298  18355.284202 -18110.638202  244.646001  19285.562286 -19041.268376   
299  18356.021770 -18111.253756  244.768032  19268.867211 -19027.882277   

                 
             kl  
0    267.415786  
1    347.939173  
2    372.670750  
3    378.067958  
4    373.154231  
..          ...  
295  243.499998  
296  243.041009  
297  243.799060  
298  244.293935  
299  240.984892  

[300 rows x 6 columns]

In [12]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [13]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [14]:
sum(trapallada.util.get_layers_n_parameters(model).values())

2593541

Data loaders for both the training and validation sets

In [15]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

A batch from the **validation** set

In [16]:
# batch = next(iter(train_dl))
batch = next(iter(validation_dl))

In [17]:
# ds.metadata_df

An element from the above batch is encoded, decoded, and a prediction plotted

In [18]:
# i = 0
# i = 1
i = 25

i_s, x_s, y_s = batch
x_s = x_s.numpy()
x_pred, z_mean, z_sd = model(x_s[i], jr.PRNGKey(42))

display(ds.metadata_df.loc[i_s[i].item()])

# first dimension of `x_pred` is 1, and hence it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': x_s[i].ravel()})
n_rows = 2000
alt.Chart(
    df.iloc[:n_rows].reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

label                                 FM
SNR                                3.162
color                               None
file name    FM/FM_SNR3.162_Simul664.mat
Name: 2552, dtype: object

alt.Chart(...)

In [19]:
z_mean, z_sd

(Array([ 9.0892416e-01, -3.0295203e+00,  3.5027260e-01, -1.6428545e+00,
        -1.2641844e+00, -3.1973532e-01,  1.2142084e-01,  4.0501618e-01,
         2.3701215e+00,  2.8033969e+00,  3.2781267e+00,  3.5931563e-01,
        -1.1001350e+00, -1.0564542e+00, -1.2125755e-01,  9.7100502e-01,
         1.4496276e-01,  3.6949043e+00, -1.3217435e+00,  2.8191623e-01,
         1.6639142e+00, -1.7853751e+00,  1.5532608e+00, -2.0609162e+00,
         2.0465236e+00, -3.6297252e+00, -1.7647793e+00, -3.0640063e+00,
        -2.6856375e+00,  1.4834256e-01,  1.3747585e+00, -1.8901572e+00,
         1.6490878e+00,  2.7129604e-03, -1.9139279e+00,  3.2008395e+00,
         5.0912285e-01,  5.7847059e-01, -2.7121079e+00,  1.4503566e+00,
         3.9448309e+00, -3.0454108e-04, -2.4948647e+00, -3.2152057e+00,
        -5.4616588e-01, -5.8858854e-01,  4.9011266e-01,  5.6632823e-01,
         2.3385857e-01, -1.0792105e-02], dtype=float32),
 Array([0.02639751, 0.03378533, 0.03002854, 0.03463432, 0.02817997,
        0.0

The whole **validation** set is encoded, and the corresponding indexes in the full datset recorded

In [20]:
z_s, i_s = [], []
for i, x, y in validation_dl:
    z_s.append(jax.vmap(model.encoder)(x.numpy()))
    i_s.append(i)

All the values are (vertically) stacked together, and the standard deviation of `z` is dropped

In [21]:
# for the sake of readability
dim_z = parameters['model']['dim_z']

z_s = jnp.vstack(z_s)[:, :dim_z]
# labels = torch.concat(labels)
# sds = torch.concat(sds)
# z_s.shape, labels.shape, sds.shape

In [22]:
n_components_t_sne = 2
t_sne_embeddings = TSNE(n_components=n_components_t_sne, perplexity=30, random_state=42).fit_transform(z_s)

It'll be used a couple of times

In [23]:
z_s_names = [f'z_{i}' for i in range(1, 1 + dim_z)]
z_s_names[-2:]

['z_49', 'z_50']

Same for t-SNE

In [24]:
t_sne_names = [f'tsne_{i}' for i in range(1, 1 + n_components_t_sne)]
t_sne_names

['tsne_1', 'tsne_2']

In [25]:
# ds.metadata_df.loc[i_s]
metadata_df = ds.metadata_df.loc[torch.hstack(i_s)]
metadata_df

,label,SNR,color,file name
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat
...,...,...,...,...
3572,FM,3.162,None,FM/FM_SNR3.162_Simul69.mat
8631,carrier,10.000,None,singlePeriod/carrier_SNR10_Simul732.mat
348,FM,1.000,None,FM/FM_SNR1_Simul708.mat
2895,FM,31.620,None,FM/FM_SNR31.62_Simul787.mat


In [26]:
t_sne_embeddings

array([[  8.662113 ,  18.811094 ],
       [-28.145939 ,   3.9070683],
       [  5.789162 , -37.197994 ],
       ...,
       [ -5.436456 ,   7.0182767],
       [ 12.989974 ,  31.419943 ],
       [ 18.20472  ,  59.91295  ]], dtype=float32)

In [27]:
# tsne_df = pd.DataFrame(t_sne_embeddings, columns=t_sne_names, index=metadata_df.index)
z_tsne_df = pd.DataFrame(np.hstack((z_s, t_sne_embeddings)), columns=z_s_names + t_sne_names, index=metadata_df.index)
z_tsne_df

,z_1,z_2,z_3,z_4,z_5,z_6,z_7,z_8,z_9,z_10,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
8940,0.955214,-0.744425,0.603479,-0.056048,-1.294133,0.475934,0.665110,1.397955,-0.056667,2.233311,...,-0.871829,-1.917214,-0.987643,-1.813413,0.163278,0.348527,0.572991,1.267724,8.662113,18.811094
7298,-1.608342,-0.478551,-0.278583,0.807158,-0.615133,-0.450265,1.623957,1.607824,1.305912,3.892334,...,0.151691,-3.208567,-2.095595,0.373494,1.330342,0.287232,2.202979,1.088996,-28.145939,3.907068
10764,0.134577,-0.942041,2.522903,2.209447,-0.619013,-1.334488,-0.339012,2.424400,-0.848908,3.370028,...,0.288943,-2.610679,-0.728518,-1.346779,-0.906536,1.157483,1.510882,0.788501,5.789162,-37.197994
1331,-0.118946,-1.463893,-0.497641,1.671303,-1.558171,-0.862737,0.126494,2.153807,0.108028,3.731916,...,-2.089194,-1.416743,-0.072907,-1.834596,0.990254,1.905218,0.553023,-0.251939,2.067066,-2.760794
10608,0.151970,-0.215203,1.362321,0.446605,-2.421160,2.235139,-0.956693,1.789623,0.841463,3.596689,...,-2.658431,-5.135908,-1.863745,-1.554168,0.882697,1.772723,1.167415,0.876318,-42.801983,-45.259216
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3572,0.591913,-1.773143,0.008647,-0.177884,-1.553452,-0.792176,0.328827,0.989915,1.649924,3.474142,...,-1.416764,-2.989493,-0.668564,-0.340901,-0.213614,1.418511,1.299010,-0.296104,-22.654314,-13.188723
8631,0.785856,-1.137123,1.468219,-0.060866,-1.055785,-0.624253,-0.581929,1.287727,2.669964,4.024937,...,0.012541,-1.651167,-1.988496,-2.698798,1.612636,-1.271968,-0.514034,0.533330,7.438111,63.627171
348,1.623707,-1.252999,-0.098444,1.228226,0.199356,-0.328901,1.305778,2.614198,-0.388743,2.070029,...,-3.020882,-3.747573,-1.675923,-2.436571,0.289195,1.142519,-0.459211,0.508918,-5.436456,7.018277
2895,0.871048,0.653092,3.945872,1.251127,2.229701,1.723474,-0.396460,1.543021,5.413857,1.321250,...,-1.877997,2.839047,2.583745,3.877289,1.637057,2.425468,-0.898426,-2.282325,12.989974,31.419943


In [28]:
# pd.concat([metadata_df, pd.DataFrame(t_sne_embeddings, columns=t_sne_names)], axis=1)
df = pd.concat([metadata_df, z_tsne_df], axis=1)
df.head()

,label,SNR,color,file name,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
8940,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul931.mat,0.955214,-0.744425,0.603479,-0.056048,-1.294133,0.475934,...,-0.871829,-1.917214,-0.987643,-1.813413,0.163278,0.348527,0.572991,1.267724,8.662113,18.811094
7298,AM,31.620,None,AM/AM_SNR31.62_Simul641.mat,-1.608342,-0.478551,-0.278583,0.807158,-0.615133,-0.450265,...,0.151691,-3.208567,-2.095595,0.373494,1.330342,0.287232,2.202979,1.088996,-28.145939,3.907068
10764,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul853.mat,0.134577,-0.942041,2.522903,2.209447,-0.619013,-1.334488,...,0.288943,-2.610679,-0.728518,-1.346779,-0.906536,1.157483,1.510882,0.788501,5.789162,-37.197994
1331,FM,31.620,None,FM/FM_SNR31.62_Simul524.mat,-0.118946,-1.463893,-0.497641,1.671303,-1.558171,-0.862737,...,-2.089194,-1.416743,-0.072907,-1.834596,0.990254,1.905218,0.553023,-0.251939,2.067066,-2.760794
10608,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul60.mat,0.151970,-0.215203,1.362321,0.446605,-2.421160,2.235139,...,-2.658431,-5.135908,-1.863745,-1.554168,0.882697,1.772723,1.167415,0.876318,-42.801983,-45.259216


In [29]:
df.dtypes

label              category
SNR                 float64
color                object
file name    string[python]
z_1                 float32
z_2                 float32
z_3                 float32
z_4                 float32
z_5                 float32
z_6                 float32
z_7                 float32
z_8                 float32
z_9                 float32
z_10                float32
z_11                float32
z_12                float32
z_13                float32
z_14                float32
z_15                float32
z_16                float32
z_17                float32
z_18                float32
z_19                float32
z_20                float32
z_21                float32
z_22                float32
z_23                float32
z_24                float32
z_25                float32
z_26                float32
z_27                float32
z_28                float32
z_29                float32
z_30                float32
z_31                float32
z_32                

Signals that are just noise have no SNR

In [30]:
df['SNR'].isna().sum()

197

In [31]:
# df.loc[df['SNR'].isna(), 'SNR'] = df['SNR'].max()

In [32]:
# df = pd.DataFrame(np.hstack((z_s, t_sne_embeddings)), columns=z_s_names + t_sne_names)
# # df['label'] = [dict(ds.labels_names)[e] for e in labels.tolist()]
# # df['label'] = df['label'].astype('category')
# # df['std'] = sds
# # df['file name'] = ds.labels_df['file name']
# df

In [33]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)